In [ ]:
#!/usr/bin/env python3
"""
Project 13 -- 1M-scBloodNL paired DE, SPLIT BY STIMULATION CONDITION (v2).
Brian, 2026-07-18. Supersedes the pooled-stim DE in kaggle_1m_scbloodnl_de.py
FOR THE INTERVENTIONAL TEST ONLY. Does NOT overwrite v1 outputs.

WHY THIS EXISTS
---------------
v1 defined, explicitly and defensibly:
    stim = ALL non-UT cells pooled (any of 3hCA/24hCA/3hPA/24hPA/3hMTB/24hMTB)
That was the right call for Gate 2 (it mirrored Calderon's single generic
stimulated state). But it collapses SIX distinct perturbations into one binary,
leaving 4 response columns total -- one per cell type. The interventional test
(notebooks/kaggle_interventional_causal_vs_coexpression.py) correlates
perturbation-RESPONSE PROFILES across conditions. A 4-point profile is not a
test; it is noise with a p-value.

This v2 changes exactly one thing: the bucket key becomes
    (group, donor, timepoint)   instead of   (group, donor, is_stim)
and each stim condition is tested against that same donor's UT baseline.
Everything else -- streaming aggregation, gene-symbol combination across
chemistries, paired t-test, BH correction -- is v1's, unchanged.

Result: 4 cell types x 6 conditions = up to 24 response columns instead of 4.
That is the difference between a real profile correlation and a withheld verdict.
No new data, no new download, same pipeline.

WHAT IS AND IS NOT REVISITED
----------------------------
- Cell-type grouping: FROZEN, identical to v1 (Calderon mapping).
- Donor pairing: now per (group, condition) -- a donor must have UT AND that
  specific condition. Donor counts will be LOWER per condition than v1's pooled
  104, because pooling let any stim sample qualify. This is expected and is
  reported per condition; if a condition drops below MIN_DONORS it is skipped
  rather than silently reported at low power.
- Statistics: unchanged (paired t across donors, BH within each group x condition).

OUTPUT (new filenames -- v1 outputs are NOT touched)
----------------------------------------------------
    /kaggle/working/scbloodnl_de_percond_<group>.parquet
        columns: group, condition, gene, n_donors_used, baseline_mean,
                 mean_diff_stim_minus_baseline, p, q
        (all conditions for that cell type stacked in one file)
    /kaggle/working/scbloodnl_de_percond_report.json

Bring these back into data/processed/ alongside the v1 files, then run the
interventional test -- it splits on group x condition automatically.

MEMORY NOTE: buckets go from ~832 to ~2,900, so the accumulators grow ~3.5x
(roughly 0.5 GB per matrix, ~3 GB peak). Fine on Kaggle; would not be fine in
the sandbox, which is one more reason this is a Kaggle job.
"""
from __future__ import annotations
import gzip, json, time, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

IS_KAGGLE = Path("/kaggle").exists()
WORK = Path("/kaggle/working") if IS_KAGGLE else Path(__file__).resolve().parent
RAW = WORK / "raw_1m_scbloodnl"
RAW.mkdir(parents=True, exist_ok=True)

BASE_URL = "https://molgenis26.gcc.rug.nl/downloads/1m-scbloodnl"

# FROZEN -- identical to v1. Do not edit without routing to Brian.
GROUP_MAP = {
    "B": "B",
    "mono 1": "monocyte", "mono 2": "monocyte", "mono 3": "monocyte", "mono 4": "monocyte",
    "memory CD8T": "CD8T", "naive CD8T": "CD8T",
    "naive CD4T": "naive_CD4T",
}
GROUPS = ["B", "monocyte", "CD8T", "naive_CD4T"]
BASELINE = "UT"
MIN_DONORS = 20          # below this, a condition is skipped, not quietly reported
CHUNK_ROWS = 5_000_000


def fetch(fname, dest=None):
    dest = dest or (RAW / fname)
    if dest.exists() and dest.stat().st_size > 0:
        print(f"[fetch] {fname} already present ({dest.stat().st_size/1e6:.1f}MB), skip")
        return dest
    url = f"{BASE_URL}/{fname}"
    print(f"[fetch] downloading {url} ...")
    t0 = time.time()
    urllib.request.urlretrieve(url, dest)
    print(f"[fetch] done {fname}: {dest.stat().st_size/1e6:.1f}MB in {time.time()-t0:.1f}s")
    return dest


def build_bucket_map():
    """Barcode -> (group, donor, timepoint). THE ONE SUBSTANTIVE CHANGE FROM v1."""
    ct = pd.read_csv(fetch("1M_cell_types.tsv"), sep="\t")
    asn = pd.read_csv(fetch("1M_assignments_conditions_expid.tsv"), sep="\t")
    meta = ct.merge(asn, on="barcode", how="inner")

    meta["group"] = meta.cell_type.map(GROUP_MAP)
    meta = meta.dropna(subset=["group", "assignment", "timepoint"]).copy()
    meta["donor"] = meta.assignment.astype(int)
    meta["cond"] = meta.timepoint.astype(str)          # UT, 3hCA, 24hCA, 3hPA, ...

    conds = sorted(meta.cond.unique())
    stim_conds = [c for c in conds if c != BASELINE]
    print(f"[cond] observed timepoints: {conds}")
    print(f"[cond] baseline='{BASELINE}', {len(stim_conds)} stim conditions: {stim_conds}")
    if not stim_conds:
        raise SystemExit("FATAL: no non-baseline conditions found -- check the timepoint column.")

    # keep only donors that have a UT baseline in that cell type (pairing anchor)
    has_ut = meta[meta.cond == BASELINE].groupby("group").donor.unique().to_dict()
    meta = meta[[d in set(has_ut.get(g, [])) for g, d in zip(meta.group, meta.donor)]].copy()

    meta["bucket_str"] = meta.group + "|" + meta.donor.astype(str) + "|" + meta.cond
    buckets = sorted(meta.bucket_str.unique())
    bucket_id = {b: i + 1 for i, b in enumerate(buckets)}   # 1-indexed, 0 = unwanted
    meta["bucket_id"] = meta.bucket_str.map(bucket_id)

    bucket_meta = pd.DataFrame(
        [(bid, *b.split("|")) for b, bid in bucket_id.items()],
        columns=["bucket_id", "group", "donor", "cond"]
    )
    bucket_meta["donor"] = bucket_meta.donor.astype(int)
    print(f"[bucket] {len(buckets)} buckets (v1 had 832) | donors per group: "
          f"{bucket_meta.groupby('group').donor.nunique().to_dict()}")
    return meta.set_index("barcode")["bucket_id"], bucket_meta, stim_conds


def col_bucket_array(barcode_bucket, chem_barcodes_path):
    bcs = pd.read_csv(chem_barcodes_path, header=None)[0]
    arr = bcs.map(barcode_bucket).fillna(0).astype(np.int64).values
    print(f"[colbucket] {chem_barcodes_path.name}: {len(bcs)} cells, {int((arr>0).sum())} wanted")
    return arr


def stream_aggregate(mtx_path, col_bucket, n_genes, n_buckets):
    """Returns (sum_mat, cnt_mat): (n_genes, n_buckets). Unchanged from v1."""
    sum_mat = np.zeros((n_genes, n_buckets + 1), dtype=np.float64)
    cnt_mat = np.zeros((n_genes, n_buckets + 1), dtype=np.int64)
    print(f"[mem] accumulators for {mtx_path.name}: "
          f"{(sum_mat.nbytes + cnt_mat.nbytes)/1e9:.2f} GB")
    t0 = time.time()
    with gzip.open(mtx_path, "rt") as f:
        f.readline()
        dims = f.readline().split()
        decl_nnz = int(dims[2])
        print(f"[stream] {mtx_path.name}: {dims[0]} genes x {dims[1]} cells, {decl_nnz} nnz")
        reader = pd.read_csv(f, sep=" ", header=None, names=["gene", "cell", "val"],
                             dtype={"gene": np.int32, "cell": np.int32, "val": np.float64},
                             chunksize=CHUNK_ROWS)
        n_rows = 0
        for chunk in reader:
            b = col_bucket[chunk["cell"].values - 1]
            keep = b > 0
            if keep.any():
                g = chunk["gene"].values[keep] - 1
                np.add.at(sum_mat, (g, b[keep]), chunk["val"].values[keep])
                np.add.at(cnt_mat, (g, b[keep]), 1)
            n_rows += len(chunk)
            if n_rows % (CHUNK_ROWS * 5) == 0:
                print(f"  [stream] {n_rows}/{decl_nnz} rows ({time.time()-t0:.0f}s)")
    print(f"[stream] {mtx_path.name} done: {n_rows} rows in {time.time()-t0:.0f}s")
    return sum_mat[:, 1:], cnt_mat[:, 1:]


def process_chemistry(chem, barcode_bucket, n_buckets):
    features_path = fetch(f"10x_{chem}_SCT_features.tsv.gz")
    barcodes_path = fetch(f"10x_{chem}_barcodes.tsv.gz")
    matrix_path = fetch(f"10x_{chem}_SCT_matrix.mtx.gz")
    genes = pd.read_csv(features_path, header=None)[0].values
    col_bucket = col_bucket_array(barcode_bucket, barcodes_path)
    sum_mat, cnt_mat = stream_aggregate(matrix_path, col_bucket, len(genes), n_buckets)
    return genes, sum_mat, cnt_mat


def main():
    t0 = time.time()
    barcode_bucket, bucket_meta, stim_conds = build_bucket_map()
    n_buckets = int(bucket_meta.bucket_id.max())

    genes_v2, sum_v2, cnt_v2 = process_chemistry("v2", barcode_bucket, n_buckets)
    genes_v3, sum_v3, cnt_v3 = process_chemistry("v3", barcode_bucket, n_buckets)

    all_genes = sorted(set(genes_v2) | set(genes_v3))
    gidx = {g: i for i, g in enumerate(all_genes)}
    n_all = len(all_genes)
    sum_c = np.zeros((n_all, n_buckets), dtype=np.float64)
    cnt_c = np.zeros((n_all, n_buckets), dtype=np.int64)
    for genes, sm, cm in [(genes_v2, sum_v2, cnt_v2), (genes_v3, sum_v3, cnt_v3)]:
        idx_map = np.array([gidx[g] for g in genes])
        np.add.at(sum_c, idx_map, sm)
        np.add.at(cnt_c, idx_map, cm)
    del sum_v2, cnt_v2, sum_v3, cnt_v3
    print(f"[combine] {n_all} genes (v2={len(genes_v2)}, v3={len(genes_v3)}) ({time.time()-t0:.0f}s)")

    mean_mat = np.divide(sum_c, cnt_c, out=np.full_like(sum_c, np.nan), where=cnt_c > 0)

    report = {"n_all_genes": n_all, "n_buckets": n_buckets,
              "baseline": BASELINE, "stim_conditions": stim_conds,
              "min_donors": MIN_DONORS, "groups": {}, "skipped": []}

    for group in GROUPS:
        gb = bucket_meta[bucket_meta.group == group]
        base_bucket = {int(r.donor): int(r.bucket_id) for r in gb[gb.cond == BASELINE].itertuples()}
        frames, ginfo = [], {}

        for cond in stim_conds:
            stim_bucket = {int(r.donor): int(r.bucket_id) for r in gb[gb.cond == cond].itertuples()}
            donors = sorted(set(base_bucket) & set(stim_bucket))
            if len(donors) < MIN_DONORS:
                print(f"[DE] {group}/{cond}: SKIP -- only {len(donors)} paired donors (< {MIN_DONORS})")
                report["skipped"].append({"group": group, "condition": cond, "n_donors": len(donors)})
                continue

            base_vals = mean_mat[:, [base_bucket[d] - 1 for d in donors]]
            stim_vals = mean_mat[:, [stim_bucket[d] - 1 for d in donors]]
            valid = ((~np.isnan(base_vals)).sum(axis=1) >= 3) & ((~np.isnan(stim_vals)).sum(axis=1) >= 3)

            diffs = stim_vals - base_vals
            n_used = (~np.isnan(diffs)).sum(axis=1)
            mean_diff = np.nanmean(diffs, axis=1)
            sd_diff = np.nanstd(diffs, axis=1, ddof=1)

            pvals = np.ones(n_all)
            with np.errstate(invalid="ignore", divide="ignore"):
                se = sd_diff / np.sqrt(n_used)
                tstat = mean_diff / se
                ok = valid & (se > 0) & np.isfinite(tstat)
                pvals[ok] = 2 * stats.t.sf(np.abs(tstat[ok]), df=n_used[ok] - 1)
            pvals[~np.isfinite(pvals)] = 1.0

            qvals = np.ones(n_all)
            if valid.sum() > 0:
                qvals[valid] = multipletests(pvals[valid], method="fdr_bh")[1]

            df = pd.DataFrame({
                "group": group, "condition": cond, "gene": all_genes,
                "n_donors_used": n_used,
                "baseline_mean": np.nanmean(base_vals, axis=1),
                "mean_diff_stim_minus_baseline": mean_diff,
                "p": pvals, "q": qvals,
            })[valid].reset_index(drop=True)
            frames.append(df)

            n_sig = int((df.q < 0.05).sum())
            ginfo[cond] = {"n_donors_paired": len(donors), "n_genes_tested": int(valid.sum()),
                           "n_sig_q05": n_sig, "n_nonsig_q05": int(valid.sum()) - n_sig}
            print(f"[DE] {group}/{cond}: donors={len(donors)} genes={int(valid.sum())} sig={n_sig}")

        if frames:
            out_path = WORK / f"scbloodnl_de_percond_{group}.parquet"
            pd.concat(frames, ignore_index=True).to_parquet(out_path)
            print(f"[DE] {group}: {len(frames)} conditions -> {out_path}")
        report["groups"][group] = ginfo

    n_cols = sum(len(v) for v in report["groups"].values())
    report["n_response_columns"] = n_cols
    report["total_runtime_s"] = time.time() - t0
    with open(WORK / "scbloodnl_de_percond_report.json", "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n[done] {n_cols} response columns (v1 pooled gave 4) in {report['total_runtime_s']:.0f}s")
    if n_cols < 8:
        print("WARNING: fewer response columns than hoped -- check the SKIP lines above. "
              "Profile correlation stays weak; report this to Brian before running the "
              "interventional test rather than running it underpowered.")
    print(json.dumps(report, indent=2))


if __name__ == "__main__":
    main()